In [1]:
import os
import sys

ROOT_PATH = '../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [2]:
import torch
from peft import LoraConfig, get_peft_model
from ultralytics.utils.loss import v8SegmentationLoss, E2ELoss
from torch.nn import L1Loss
from torch.optim import AdamW

from datasets.rescuenet_dataset import RescueNetDataset, collate_fn
from utils.augmentations import get_augmentation_pipeline
from model.TriheadSegmentationModel import TriheadSegmentationModel
from model.UncertaintyLossWeighting import UncertaintyLossWeighting
from utils.training import compute_normal_loss, EarlyStoppingAndCheckpointing

torch.cuda.empty_cache()
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device('cpu')
print(device)

cpu


In [3]:
triheadSegmentationModel = TriheadSegmentationModel(
  yolo_pt_path="yolo26n-sem.pt",
  device=device,
  # include_depth=False,
  # include_normals=False,
)
target_modules = triheadSegmentationModel.yolo.modules()
loss_balancer = UncertaintyLossWeighting()

In [4]:
valid_target_modules = []
for name, module in triheadSegmentationModel.yolo.model.named_modules():
  if isinstance(module, torch.nn.Conv2d):
    if module.groups == 1:
      valid_target_modules.append(name)

In [5]:
spatial_aug, photometric_aug = get_augmentation_pipeline()

In [6]:
# vanilla_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
# vanilla_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=False)
# vanilla_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=False)
# vanilla_train_loader = torch.utils.data.DataLoader(vanilla_train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
# vanilla_val_loader = torch.utils.data.DataLoader  (vanilla_val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
# vanilla_test_loader = torch.utils.data.DataLoader (vanilla_test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
# print(f'Train Count: {len(vanilla_train_dataset)}')
# print(f'Validation Count: {len(vanilla_val_dataset)}')
# print(f'Test Count: {len(vanilla_test_dataset)}')

In [7]:
# only_normals_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
# only_normals_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=False)
# only_normals_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=False)
# only_normals_train_loader = torch.utils.data.DataLoader(only_normals_train_dataset, batch_size=16, shuffle=True)
# only_normals_val_loader = torch.utils.data.DataLoader(only_normals_val_dataset, batch_size=16, shuffle=False)
# only_normals_test_loader = torch.utils.data.DataLoader(only_normals_test_dataset, batch_size=16, shuffle=False)
# print(f'Train Count: {len(only_normals_train_dataset)}')
# print(f'Validation Count: {len(only_normals_val_dataset)}')
# print(f'Test Count: {len(only_normals_test_dataset)}')

In [8]:
# only_depth_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_normals=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
# only_depth_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_normals=False)
# only_depth_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_normals=False)
# only_depth_train_loader = torch.utils.data.DataLoader(only_depth_train_dataset, batch_size=16, shuffle=True)
# only_depth_val_loader = torch.utils.data.DataLoader(only_depth_val_dataset, batch_size=16, shuffle=False)
# only_depth_test_loader = torch.utils.data.DataLoader(only_depth_test_dataset, batch_size=16, shuffle=False)
# print(f'Train Count: {len(only_depth_train_dataset)}')
# print(f'Validation Count: {len(only_depth_val_dataset)}')
# print(f'Test Count: {len(only_depth_test_dataset)}')

In [9]:
tri_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', spatial_transform=spatial_aug, photometric_transform=photometric_aug)
tri_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val')
tri_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test')
tri_train_loader = torch.utils.data.DataLoader(tri_train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
tri_val_loader = torch.utils.data.DataLoader(tri_val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
tri_test_loader = torch.utils.data.DataLoader(tri_test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
print(f'Train Count: {len(tri_train_dataset)}')
print(f'Validation Count: {len(tri_val_dataset)}')
print(f'Test Count: {len(tri_test_dataset)}')

Train Count: 3595
Validation Count: 449
Test Count: 450


In [10]:
from types import SimpleNamespace

current_args = triheadSegmentationModel.yolo.model.args if isinstance(triheadSegmentationModel.yolo.model.args, dict) else {}

if 'overlap_mask' not in current_args:
  current_args['overlap_mask'] = True

current_args['box'] = 7.5
current_args['cls'] = 0.5
current_args['dfl'] = 1.5
  
triheadSegmentationModel.yolo.model.args = SimpleNamespace(**current_args)

In [11]:
lora_config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=valid_target_modules,
  modules_to_save=["model.23.*"],
  bias="none",
)
peft_backbone = get_peft_model(triheadSegmentationModel.yolo.model, peft_config=lora_config).to(device)
peft_backbone.print_trainable_parameters()
peft_backbone.to(device)
triheadSegmentationModel.yolo.model = peft_backbone
final_model = triheadSegmentationModel.to(device)

trainable params: 1,368,496 || all params: 28,480,568 || trainable%: 4.8050


In [12]:
trainable_params = [p for p in final_model.parameters() if p.requires_grad]
trainable_params.extend(loss_balancer.parameters())

optimizer = AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

raw_yolo_architecture = final_model.yolo.model.base_model.model

seg_loss_criterion = E2ELoss(raw_yolo_architecture, loss_fn=v8SegmentationLoss)
depth_loss_criterion = L1Loss()

In [13]:
# for batch_images, batch_targets in vanilla_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [14]:
# for batch_images, batch_targets in only_depth_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [15]:
# for batch_images, batch_targets in only_normals_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [18]:
early_stopping = EarlyStoppingAndCheckpointing()

for batch_images, batch_targets in tri_train_loader:
  optimizer.zero_grad()
  segmentation_out, depth_out, normal_out = final_model(batch_images.to(device=device))
  
  depth_out = depth_out.to(device=device)
  normal_out = normal_out.to(device=device)
  # print(batch_targets)
  true_segmentation_map = batch_targets[0]
  true_depth_map = batch_targets[1]['depth'].to(device=device)
  true_surface_normals = batch_targets[2]['normals'].to(device=device)

  seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
  
  depth_loss = depth_loss_criterion(depth_out, true_depth_map)
  normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
  loss_total = (torch.abs(loss_balancer.alpha).to(device) * seg_loss) + \
                (torch.abs(loss_balancer.beta).to(device) * depth_loss) + \
                (torch.abs(loss_balancer.gamma).to(device) * normal_loss)
  loss_total = loss_total.mean()
  loss_total.backward()
  
  optimizer.step()
  
  break